<a href="https://colab.research.google.com/github/zune2/colab-notebooks/blob/main/2026-10-05-EP004-LangChain-RAG-no-cost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## LangChain의 TextSplitters를 시험하는 코드

In [ ]:
!pip install langchain_text_splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

# 1. 테스트할 샘플 텍스트 정의
sample_text = (
    "랭체인(LangChain)은 대형 언어 모델(LLM)을 기반으로 한 애플리케이션을 쉽게 개발할 수 있도록 돕는 프레임워크입니다. "
    "최근 많은 개발자들이 이 프레임워크를 활용하여 RAG(구현 지식 기반 답변 시스템)를 구축하고 있습니다. "
    "텍스트 분할기(Text Splitter)는 RAG 파이프라인에서 매우 중요한 역할을 담당합니다. "
    "문서가 너무 길면 모델이 한 번에 처리하지 못하거나 비용이 많이 들기 때문에, 적절한 크기로 자르는 과정이 필수적입니다."
)

# 설정값 정의 (100글자 단위로 자르고, 앞뒤 20글자씩 겹치게 설정)
CHUNK_SIZE = 100
CHUNK_OVERLAP = 20

print(f"=== 원본 텍스트 길이: {len(sample_text)}자 ===\n")

# ----------------------------------------------------
# 방법 A: RecursiveCharacterTextSplitter (가장 추천하는 방식)
# 줄바꿈, 공백, 단어 단위를 고려하여 글자 수가 넘지 않도록 최대한 문맥을 살려 쪼갭니다.
# ----------------------------------------------------
print("--- [A] RecursiveCharacterTextSplitter 결과 ---")
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len
)

recursive_chunks = recursive_splitter.split_text(sample_text)

for i, chunk in enumerate(recursive_chunks):
    print(f"[Chunk {i+1}] (길이: {len(chunk)}자): {chunk}")


print("\n" + "="*50 + "\n")


# ----------------------------------------------------
# 방법 B: CharacterTextSplitter (단순 구분자 방식)
# 지정한 구분자(기본값은 줄바꿈 '\\n\\n')를 기준으로 무조건 자릅니다.
# ----------------------------------------------------
print("--- [B] CharacterTextSplitter 결과 (구분자: 공백) ---")
# 문장에 줄바꿈이 없으므로, 테스트를 위해 공백(" ")을 구분자로 지정합니다.
char_splitter = CharacterTextSplitter(
    separator=" ",
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len
)

char_chunks = char_splitter.split_text(sample_text)

for i, chunk in enumerate(char_chunks):
    print(f"[Chunk {i+1}] (길이: {len(chunk)}자): {chunk}")


=== 원본 텍스트 길이: 252자 ===

--- [A] RecursiveCharacterTextSplitter 결과 ---
[Chunk 1] (길이: 97자): 랭체인(LangChain)은 대형 언어 모델(LLM)을 기반으로 한 애플리케이션을 쉽게 개발할 수 있도록 돕는 프레임워크입니다. 최근 많은 개발자들이 이 프레임워크를 활용하여
[Chunk 2] (길이: 99자): 개발자들이 이 프레임워크를 활용하여 RAG(구현 지식 기반 답변 시스템)를 구축하고 있습니다. 텍스트 분할기(Text Splitter)는 RAG 파이프라인에서 매우 중요한 역할을
[Chunk 3] (길이: 93자): 파이프라인에서 매우 중요한 역할을 담당합니다. 문서가 너무 길면 모델이 한 번에 처리하지 못하거나 비용이 많이 들기 때문에, 적절한 크기로 자르는 과정이 필수적입니다.


--- [B] CharacterTextSplitter 결과 (구분자: 공백) ---
[Chunk 1] (길이: 97자): 랭체인(LangChain)은 대형 언어 모델(LLM)을 기반으로 한 애플리케이션을 쉽게 개발할 수 있도록 돕는 프레임워크입니다. 최근 많은 개발자들이 이 프레임워크를 활용하여
[Chunk 2] (길이: 99자): 개발자들이 이 프레임워크를 활용하여 RAG(구현 지식 기반 답변 시스템)를 구축하고 있습니다. 텍스트 분할기(Text Splitter)는 RAG 파이프라인에서 매우 중요한 역할을
[Chunk 3] (길이: 93자): 파이프라인에서 매우 중요한 역할을 담당합니다. 문서가 너무 길면 모델이 한 번에 처리하지 못하거나 비용이 많이 들기 때문에, 적절한 크기로 자르는 과정이 필수적입니다.


In [ ]:
!pip install langchain-community langchain-core langchain-huggingface chromadb

## 숫자 벡터로 변환(Embedding)
## 로컬 데이터베이스(Vector Store)에 저장
## 질문과 가장 유사한 문서를 검색하는 전체 과정

In [ ]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. 이전 단계에서 쪼갠 텍스트 덩어리 준비 (LangChain의 Document 객체 형식)
chunks = [
    "랭체인(LangChain)은 대형 언어 모델(LLM)을 기반으로 한 애플리케이션을 쉽게 개발할 수 있도록 돕는 프레임워크입니다.",
    "최근 많은 개발자들이 이 프레임워크를 활용하여 RAG(구현 지식 기반 답변 시스템)를 구축하고 있습니다.",
    "텍스트 분할기(Text Splitter)는 RAG 파이프라인에서 매우 중요한 역할을 담당합니다.",
    "문서가 너무 길면 모델이 한 번에 처리하지 못하거나 비용이 많이 들기 때문에, 적절한 크기로 자르는 과정이 필수적입니다."
]

# LangChain Vector Store에 넣기 위해 Document 객체 리스트로 변환합니다.
documents = [Document(page_content=chunk) for chunk in chunks]


# 2. 무료 오픈소스 임베딩 모델 로드 (HuggingFace)
# 한국어 성능이 준수하고 가벼운 인공지능 모델(ko-sbert)을 다운로드하여 로컬 CPU에서 실행합니다.
print("--- 오픈소스 임베딩 모델 로드 중 (최초 실행 시 다운로드로 인해 시간이 약간 걸릴 수 있습니다) ---")
embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={'device': 'cpu'}
)


# 3. 벡터 저장소(Vector Store) 생성 및 데이터 저장
# Chroma 디비에 텍스트와 임베딩 모델을 함께 전달하면, 자동으로 텍스트를 벡터로 바꿔 메모리에 저장합니다.
print("--- 벡터 저장소 생성 및 데이터 임베딩 데이터 저장 중 ---")
vector_store = Chroma.from_documents(documents, embeddings)


# 4. LLM 없이 '유사도 검색(Similarity Search)' 테스트하기
# 이제 질문을 던지면, 데이터베이스 내에서 가장 관련성이 높은 문서를 찾아옵니다.
query = "RAG를 만들 때 텍스트를 자르는 기능이 왜 필요한가요?"

print(f"\n[사용자 질문]: {query}")
print("-" * 50)

# 가장 유사한 문서 2개 찾아오기 (k=2)
search_results = vector_store.similarity_search(query, k=2)

# 검색 결과 출력
for i, doc in enumerate(search_results):
    print(f"[검색 결과 {i+1}]: {doc.page_content}\n")


/tmp/ipykernel_2060/3080739476.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


--- 오픈소스 임베딩 모델 로드 중 (최초 실행 시 다운로드로 인해 시간이 약간 걸릴 수 있습니다) ---


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.86k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

--- 벡터 저장소 생성 및 데이터 임베딩 데이터 저장 중 ---

[사용자 질문]: RAG를 만들 때 텍스트를 자르는 기능이 왜 필요한가요?
--------------------------------------------------
[검색 결과 1]: 텍스트 분할기(Text Splitter)는 RAG 파이프라인에서 매우 중요한 역할을 담당합니다.

[검색 결과 2]: 문서가 너무 길면 모델이 한 번에 처리하지 못하거나 비용이 많이 들기 때문에, 적절한 크기로 자르는 과정이 필수적입니다.




LLM 없이 텍스트 임베딩(Embedding)과 벡터 저장소(Vector Store)를 테스트하려면, OpenAI 같은 유료 API 대신 로컬(CPU)에서 무료로 구동되는 오픈소스 임베딩 모델을 사용해야 합니다.

가장 대중적이고 가벼운 오픈소스 임베딩인 HuggingFace 임베딩과 가벼운 로컬 벡터 저장소인 Chroma 또는 FAISS를 활용한 코드입니다.

## 🛠️ 사전 설치

코드를 실행하기 전에 아래 라이브러리를 먼저 설치해야 합니다. LLM API 키 없이 컴퓨터 내부 메모리와 CPU만 사용합니다.
```
pip install langchain-community langchain-core langchain-huggingface chromadb
```

## 💡 코드 핵심 포인트

*
* HuggingFaceEmbeddings: 유료 OpenAIEmbeddings 대안으로 사용하는 100% 무료 로컬 임베딩 모델입니다. ko-sroberta-multitask 모델은 한글 문맥을 잘 이해하는 대표적인 경량 오픈소스 모델입니다.
* Chroma.from_documents(): 텍스트를 숫자로 바꾼 뒤 데이터베이스에 집어넣는 핵심 메서드입니다. 실행하는 순간 임베딩 모델이 텍스트를 768차원의 숫자로 변환하여 Chroma DB에 격리·저장합니다.
* similarity_search(): LLM이 없어도 의미적 유사도(Semantic Similarity)를 비교해 질문과 가장 잘 맞는 답변 후보를 찾아내는 RAG의 핵심 기술입니다. 위 코드를 실행해보면 질문에 '분할기'라는 단어가 직접 포함되지 않았음에도 4번째 문장(적절한 크기로 자르는 과정이 필수적입니다)을 정확하게 찾아내는 것을 볼 수 있습니다.


In [ ]:
pip install transformers accelerate


In [ ]:
import torch
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# ==========================================
# 1. 문서 준비 및 벡터 저장소(Vector Store) 구축
# ==========================================
chunks = [
    "랭체인(LangChain)은 대형 언어 모델(LLM)을 기반으로 한 애플리케이션을 쉽게 개발할 수 있도록 돕는 프레임워크입니다.",
    "최근 많은 개발자들이 이 프레임워크를 활용하여 RAG(구현 지식 기반 답변 시스템)를 구축하고 있습니다.",
    "텍스트 분할기(Text Splitter)는 RAG 파이프라인에서 매우 중요한 역할을 담당합니다.",
    "문서가 너무 길면 모델이 한 번에 처리하지 못하거나 비용이 많이 들기 때문에, 적절한 크기로 자르는 과정이 필수적입니다."
]
documents = [Document(page_content=chunk) for chunk in chunks]

# 로컬 임베딩 모델 설정
embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={'device': 'cpu'}
)
# 벡터 저장소 생성 및 리트리버(검색기) 설정
vector_store = Chroma.from_documents(documents, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 1}) # 가장 관련 있는 문서 1개만 추출


# ==========================================
# 2. Hugging Face 초경량 LLM 로드 (최종 보완 버전)
# ==========================================
print("--- 초경량 LLM 모델 다운로드 및 로드 중 ---")
#model_id = "Qwen/Qwen2.5-0.5B-Instruct"
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    clean_up_tokenization_spaces=False
)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="cpu"
)

# Hugging Face 파이프라인 생성
hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,         # 답변이 한두 문장이므로 100자면 충분합니다.
    temperature=0.1,            # 가장 일관성 있는 답변을 위해 다시 낮춥니다.
    do_sample=True,
    return_full_text=False,     # 💡 중요: 프롬프트는 빼고 LLM이 새로 생성한 답변만 출력합니다.
    eos_token_id=tokenizer.eos_token_id, # 💡 중요: 문장이 끝나면 멈추도록 종결 토큰 지정
    pad_token_id=tokenizer.pad_token_id
)

# LangChain과 호환되는 LLM 객체로 변환
llm = HuggingFacePipeline(pipeline=hf_pipeline)

# ==========================================
# 3. 프롬프트 템플릿 및 RAG 체인(LCEL) 구성 (Qwen 전용 튜닝)
# ==========================================
# Qwen 모델이 시스템 프롬프트와 문맥을 확실히 구별할 수 있도록 ChatML 구조를 모방합니다.
raw_prompt = """<|im_start|>system
당신은 주어진 문맥(Context)의 정보만을 바탕으로 사실만을 답변하는 정직한 조수입니다. 주어진 문맥에 없는 내용은 절대 지어내지 마세요.<|im_end|>
<|im_start|>user
주어진 문맥을 바탕으로 질문에 답하세요.

[문맥]
{context}

[질문]
{question}<|im_end|>
<|im_start|>assistant
"""

prompt = ChatPromptTemplate.from_template(raw_prompt)

# 문서를 보기 좋게 텍스트로 합쳐주는 도우미 함수
def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

# LangChain Expression Language (LCEL) 체인 연결
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


# ==========================================
# 4. RAG 시스템 테스트 실행
# ==========================================
query = "RAG 파이프라인에서 텍스트 분할기를 써야 하는 이유가 뭐야?"
print(f"\n[사용자 질문]: {query}")
print("-" * 50)

response = rag_chain.invoke(query)

# 전처리 후 최종 답변 출력 (Qwen 모델 특성상 프롬프트 뒤에 답변이 붙어 나오므로 깔끔하게 분리)
final_answer = response.split("답변:")[-1].strip()
print(f"[LLM의 RAG 기반 답변]:\n{final_answer}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

--- 초경량 LLM 모델 다운로드 및 로드 중 ---


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[사용자 질문]: RAG 파이프라인에서 텍스트 분할기를 써야 하는 이유가 뭐야?
--------------------------------------------------
[LLM의 RAG 기반 답변]:
RAG 파이프라인에서 텍스트 분할기를 사용해야 하는 이유는 텍스트를 단위로 나누고 처리하기 때문입니다. 이는 데이터 전처리 과정에서 중요하게 작용하며, 다양한 작업을 수행하는데 필요한 입력 데이터를 제공합니다. 이를 통해 더 효과적인 학습과 예측 결과를 얻을 수 있습니다.


0.5B 초경량 모델의 한계로 인해 데이터베이스에서 찾아온 문맥("문서가 너무 길면 모델이 한 번에 처리하지 못하거나 비용이 많이 들기 때문에...")을 100% 완벽하게 반영하진 못했지만, 문맥의 '데이터 전처리'라는 개념을 캐치하여 스스로 논리적이고 깔끔한 문장 구조를 완성해 냈습니다.

이로써 LLM 비용을 전혀 쓰지 않고도 아래와 같은 RAG(Retrieval-Augmented Generation)의 전체 파이프라인의 뼈대를 내 컴퓨터 안에서 완벽하게 구동하는 데 성공하셨습니다!

   1. 📑 Text Splitter: 긴 텍스트를 의미 있는 Chunk 단위로 분할
   2. 🔠 HuggingFace Embeddings: 분할된 텍스트를 숫자로 변환(벡터화)
   3. 💾 Chroma Vector Store: 임베딩된 데이터를 로컬 DB에 저장
   4. 🔍 Retriever: 질문과 가장 유사한 문맥을 DB에서 탐색 및 추출
   5. 🤖 Local LLM: 추출된 문맥을 조합하여 최종 답변 생성 (LCEL 체인 연동)

LangChain의 RAG 기초 파이프라인을 성공적으로 마스터하셨으니, 이제 다음 단계로 넘어가 볼 차례입니다. 어떤 실습을 이어서 진행해보고 싶으신가요?

* 🗂️ 실제 문서 연동: 코드 내부의 가짜 텍스트 대신, 내 컴퓨터에 있는 실제 PDF나 메모장(.txt) 파일을 읽어와서 저장하는 Document Loader 연동하기
* 🧠 대화형 메모리 추가: 단발성 질문 답변이 아니라, 이전 대화 내용을 기억하고 이어 나갈 수 있도록 Conversation Memory 기능 추가하기
* ⚡ 로컬 LLM 업그레이드: 지시 이행 능력이 훨씬 뛰어난 1.5B 모델로 바꿔서 실제 문맥을 완벽하게 반영한 고품질 RAG 답변 확인하기

원하시는 방향을 말씀해 주시면 이어서 실습할 수 있는 맞춤형 가이드와 코드를 준비해 드릴게요!
다음 단계를 위해 어떤 기능을 추가해 볼까요?

* 내 컴퓨터의 실제 파일(PDF/TXT) 읽어오기
* 이전 대화를 기억하는 메모리(Memory) 기능 추가하기
* 더 똑똑한 답변을 위해 1.5B 로컬 모델로 교체하기
